### Activity 1: Fireworks vs OpenAI RAGAS + LangSmith Cost

Compare the Fireworks OSS RAG path vs an OpenAI `gpt-4.1-mini` path on the same cat-health questions.

**Reuse ideas from Session 5** (`05_Synthetic_Data_Generation_for_RAG_Evals/...Ragas_LangSmith.ipynb`).

Goal for a quick ship:
1. Same docs + same ~5 questions
2. Run both generators
3. Score with RAGAS (faithfulness / answer relevancy / context precision if available)
4. Trace both in LangSmith and compare tokens/$

## 0) Env checklist

Add to `.env` if missing:
- `OPENAI_API_KEY`
- `LANGCHAIN_API_KEY` (or `LANGSMITH_API_KEY`)
- `LANGCHAIN_TRACING_V2=true`
- `LANGCHAIN_PROJECT=session10-activity1`

Install (once, in this project's venv):
```bash
uv add ragas datasets langchain-openai
```
Keep Fireworks chat/embedding model vars as you used for `main.py` (`qwen3-embedding-8b`).

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "session10-activity1")

required = ["FIREWORKS_API_KEY", "OPENAI_API_KEY"]
langsmith = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")
print({k: bool(os.getenv(k)) for k in required})
print("LangSmith key set:", bool(langsmith))
assert all(os.getenv(k) for k in required), "Missing API keys in .env"
assert langsmith, "Set LANGCHAIN_API_KEY or LANGSMITH_API_KEY for cost traces"

{'FIREWORKS_API_KEY': True, 'OPENAI_API_KEY': True}
LangSmith key set: True


### 1) Add a small eval set

Eval set includes;
- 5 questions + short `ground_truth` answers from `data/
- cat-health-guide.pdf` (vaccines, nutrition, etc.)
- answerable from the PDF so faithfulness is meaningful.

In [4]:
eval_set = [
    {
        "question": "What feline life stages do the 2021 AAHA/AAFP guidelines use?",
        "ground_truth": "Kitten, young adult, mature adult, senior, and an end-of-life stage (five-stage grouping).",
    },
    {
        "question": "How often should cats have veterinary examinations according to the guidelines?",
        "ground_truth": "At least annually for all cats; senior cats at least every 6 months, and more often if they have chronic conditions.",
    },
    {
        "question": "What are the core vaccines for cats listed in the guidelines?",
        "ground_truth": "Rabies, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV).",
    },
    {
        "question": "Why do the guidelines emphasize feline-friendly handling during veterinary visits?",
        "ground_truth": "To reduce the cat's stress, improve handler safety, create a better experience for patient/client/provider, and help increase exam visit frequency and compliance.",
    },
    {
        "question": "What health issues are commonly underdiagnosed and especially important to assess in mature adult or senior cats?",
        "ground_truth": "Osteoarthritis / degenerative joint disease and pain; also quality-of-life and related exam focuses like thyroid/abdominal palpation.",
    },
]

assert len(eval_set) >= 5, "Add at least 5 questions"
print(len(eval_set), "questions ready")

5 questions ready


### 2) Two RAG runners

Reuse the retrieve→generate idea from `app/rag.py`, but make **chat model** the only intentional difference:

| Pipeline | Chat model | Embeddings (suggestion) |
|----------|------------|-------------------------|
| Fireworks | `FIREWORKS_CHAT_MODEL` / gpt-oss-20b via Fireworks base URL | Fireworks `qwen3-embedding-8b` |
| OpenAI | `gpt-4.1-mini` | OpenAI embeddings (e.g. `text-embedding-3-small`) **or** same Fireworks embeddings for a fairer generator-only compare |

Implement a function like:
```python
def run_rag(question: str, *, chat_model, embeddings) -> dict:
    # return {"answer": ..., "contexts": [str, ...]}
```

Tip: build the vector store **once per embedding backend**, then loop questions.

In [5]:
import os
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"
data_dir = os.environ.get("RAG_DATA_DIR", "data")


def _tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


# Load + split once
try:
    documents = DirectoryLoader(
        data_dir, glob="**/*.pdf", loader_cls=PyMuPDFLoader
    ).load()
except Exception:
    documents = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len
)
chunks = text_splitter.split_documents(documents) if documents else []
print(f"Loaded {len(documents)} docs → {len(chunks)} chunks")

# Shared Fireworks embeddings / retriever (generator-only compare)
embedding_model = OpenAIEmbeddings(
    model=os.environ.get(
        "FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"
    ),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=4096,
)
retriever = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    location=":memory:",
    collection_name="rag_collection",
).as_retriever()

human_template = (
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provide context to answer the provided user query. "
    "Only use the provided context to answer the query. If you do not know the answer, "
    'or it\'s not contained in the provided context respond with "I don\'t know"'
)
chat_prompt = ChatPromptTemplate.from_messages([("human", human_template)])

fw_llm = ChatOpenAI(
    model=os.environ.get(
        "FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"
    ),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
)
oa_llm = ChatOpenAI(model="gpt-4.1-mini")


def run_rag(question: str, llm, *, provider: str) -> dict:
    docs = retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    answer = (chat_prompt | llm | StrOutputParser()).invoke(
        {"query": question, "context": docs},
        config={"tags": [provider], "metadata": {"provider": provider}},
    )
    return {"answer": answer, "contexts": contexts}


# Smoke test
smoke = run_rag(eval_set[0]["question"], fw_llm, provider="fireworks")
print(smoke["answer"][:300])
print("contexts:", len(smoke["contexts"]))


/var/folders/7g/kyyns4h560zfyzjrfwb_wlkw0000gn/T/ipykernel_62871/2714819795.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


Loaded 22 docs → 42 chunks
The 2021 AAHA/AAFP Feline Life Stage Guidelines divide a cat’s lifespan into **five stages**:

1. **Kitten**  
2. **Young adult**  
3. **Mature adult**  
4. **Senior**  
5. **End‑of‑life** (the end‑of‑life stage)
contexts: 4


### 3) Collect rows for both providers

In [7]:
fireworks_rows = []
openai_rows = []

for i, item in enumerate(eval_set, start=1):
    q = item["question"]
    gt = item["ground_truth"]
    print(f"[{i}/{len(eval_set)}] {q[:60]}...")

    fw = run_rag(q, fw_llm, provider="fireworks")
    oa = run_rag(q, oa_llm, provider="openai")

    fireworks_rows.append(
        {
            "question": q,
            "answer": fw["answer"],
            "contexts": fw["contexts"],
            "ground_truth": gt,
        }
    )
    openai_rows.append(
        {
            "question": q,
            "answer": oa["answer"],
            "contexts": oa["contexts"],
            "ground_truth": gt,
        }
    )

print("Fireworks rows:", len(fireworks_rows))
print("OpenAI rows:", len(openai_rows))
print("--- Fireworks sample ---")
print(fireworks_rows[0]["answer"][:250])
print("--- OpenAI sample ---")
print(openai_rows[0]["answer"][:250])


[1/5] What feline life stages do the 2021 AAHA/AAFP guidelines use...
[2/5] How often should cats have veterinary examinations according...
[3/5] What are the core vaccines for cats listed in the guidelines...
[4/5] Why do the guidelines emphasize feline-friendly handling dur...
[5/5] What health issues are commonly underdiagnosed and especiall...
Fireworks rows: 5
OpenAI rows: 5
--- Fireworks sample ---
The 2021 AAHA/AAFP Feline Life Stage Guidelines divide a cat’s lifespan into **five stages**:  

1. **Kitten**  
2. **Young adult**  
3. **Mature adult**  
4. **Senior**  
5. **End‑of‑life** (the final phase)  

The first four are the distinct age‑re
--- OpenAI sample ---
The 2021 AAHA/AAFP Feline Life Stage Guidelines divide the cat’s lifespan into five stages with four distinct age-related stages: kitten, young adult, mature adult, and senior, as well as an end-of-life stage.


## 4) RAGAS scoring

Use metrics such as:
- Faithfulness
- Answer Relevancy
- Context Precision (and/or Context Recall)

Pattern from Session 5: build a HuggingFace/`datasets.Dataset`, then `evaluate(...)`.
Judge model can be OpenAI for both so scoring is consistent.

In [8]:
import sys
import types

# ragas 0.4.x still imports removed VertexAI symbols from langchain-community
_chat_vertex = types.ModuleType("langchain_community.chat_models.vertexai")


class ChatVertexAI:  # noqa: N801
    pass


_chat_vertex.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _chat_vertex

import langchain_community.llms as _llms

if not hasattr(_llms, "VertexAI"):

    class VertexAI:  # noqa: N801
        pass

    _llms.VertexAI = VertexAI

from datasets import Dataset
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, faithfulness

assert "fireworks_rows" in globals() and fireworks_rows, "Run Step 3 first (collect rows)"
assert "openai_rows" in globals() and openai_rows, "Run Step 3 first (collect rows)"

judge_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
judge_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def to_ragas_dataset(rows: list[dict]) -> Dataset:
    return Dataset.from_list(
        [
            {
                "user_input": r["question"],
                "response": r["answer"],
                "retrieved_contexts": r["contexts"],
                "reference": r["ground_truth"],
            }
            for r in rows
        ]
    )


metrics = [faithfulness, answer_relevancy, context_precision]
fw_ds = to_ragas_dataset(fireworks_rows)
oa_ds = to_ragas_dataset(openai_rows)
print(fw_ds)
print(oa_ds)

print("Scoring Fireworks...")
fw_scores = evaluate(
    fw_ds,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    experiment_name="session10-fireworks",
)
print(fw_scores)

print("Scoring OpenAI...")
oa_scores = evaluate(
    oa_ds,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    experiment_name="session10-openai",
)
print(oa_scores)


/var/folders/7g/kyyns4h560zfyzjrfwb_wlkw0000gn/T/ipykernel_62871/1477120004.py:28: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, context_precision, faithfulness
/var/folders/7g/kyyns4h560zfyzjrfwb_wlkw0000gn/T/ipykernel_62871/1477120004.py:28: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import answer_relevancy, context_precision, faithfulness
/var/folders/7g/kyyns4h560zfyzjrfwb_wlkw0000gn/T/ipykernel_62871/1477120004.py:28: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' i

Dataset({
    features: ['user_input', 'response', 'retrieved_contexts', 'reference'],
    num_rows: 5
})
Dataset({
    features: ['user_input', 'response', 'retrieved_contexts', 'reference'],
    num_rows: 5
})
Scoring Fireworks...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.9765, 'answer_relevancy': 0.9004, 'context_precision': 0.8056}
Scoring OpenAI...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 1.0000, 'answer_relevancy': 0.9434, 'context_precision': 0.7944}


## 5) LangSmith cost compare

Open [LangSmith](https://smith.langchain.com) → project `session10-activity1`.
Filter by tags `fireworks` vs `openai` for the RAG generator calls.

Note: RAGAS judge calls also use OpenAI (`gpt-4.1-mini`) and should **not** be counted as the Fireworks RAG cost.


In [9]:
comparison_notes = """
Quality (RAGAS) — 5 cat-health questions, shared Fireworks embeddings:

| Metric | Fireworks (gpt-oss-20b) | OpenAI (gpt-4.1-mini) |
|---|---:|---:|
| Faithfulness | 0.9765 | 1.0000 |
| Answer relevancy | 0.9004 | 0.9434 |
| Context precision | 0.8056 | 0.7944 |

Cost / tokens (LangSmith project `session10-activity1`, tagged LLM runs):
- Fireworks generator batch: ~$0.0037, ~40.1k tokens (11 LLM calls incl. smoke test)
- OpenAI generator batch: ~$0.0095, ~33.1k tokens (10 LLM calls)
- RAGAS judge (OpenAI gpt-4.1-mini, not part of either RAG path): ~$0.046, ~110k tokens

Takeaway:
- Quality is close: OpenAI slightly higher faithfulness/relevancy; Fireworks slightly higher context precision.
- For this small eval, Fireworks generator cost was lower than OpenAI while staying nearly as faithful — a solid MVP/managed-OSS tradeoff if latency/throughput remain acceptable.
"""
print(comparison_notes)

# Optional: refresh LangSmith summary from API
from datetime import datetime, timedelta, timezone
from collections import defaultdict
from dotenv import load_dotenv
import os
from langsmith import Client

load_dotenv(override=True)
client = Client()
project = os.getenv("LANGCHAIN_PROJECT", "session10-activity1")
start = datetime.now(timezone.utc) - timedelta(hours=12)
llm_runs = list(client.list_runs(project_name=project, start_time=start, run_type="llm", limit=100))

buckets = defaultdict(lambda: {"n": 0, "tokens": 0, "cost": 0.0})
for r in llm_runs:
    tags = set(r.tags or [])
    meta = (r.extra or {}).get("metadata") or {}
    provider = meta.get("provider")
    if "fireworks" in tags or provider == "fireworks":
        key = "fireworks"
    elif "openai" in tags or provider == "openai":
        key = "openai"
    else:
        key = "other_judge_or_untagged"
    buckets[key]["n"] += 1
    buckets[key]["tokens"] += r.total_tokens or 0
    cost = getattr(r, "total_cost", None)
    if cost is not None:
        buckets[key]["cost"] += float(cost)

print("LangSmith LLM summary (last 12h):")
for k, v in buckets.items():
    print(f"  {k}: n={v['n']}, tokens={v['tokens']}, cost≈${v['cost']:.6f}")
print(f"Project UI: https://smith.langchain.com → {project}")



Quality (RAGAS) — 5 cat-health questions, shared Fireworks embeddings:

| Metric | Fireworks (gpt-oss-20b) | OpenAI (gpt-4.1-mini) |
|---|---:|---:|
| Faithfulness | 0.9765 | 1.0000 |
| Answer relevancy | 0.9004 | 0.9434 |
| Context precision | 0.8056 | 0.7944 |

Cost / tokens (LangSmith project `session10-activity1`, tagged LLM runs):
- Fireworks generator batch: ~$0.0037, ~40.1k tokens (11 LLM calls incl. smoke test)
- OpenAI generator batch: ~$0.0095, ~33.1k tokens (10 LLM calls)
- RAGAS judge (OpenAI gpt-4.1-mini, not part of either RAG path): ~$0.046, ~110k tokens

Takeaway:
- Quality is close: OpenAI slightly higher faithfulness/relevancy; Fireworks slightly higher context precision.
- For this small eval, Fireworks generator cost was lower than OpenAI while staying nearly as faithful — a solid MVP/managed-OSS tradeoff if latency/throughput remain acceptable.

LangSmith LLM summary (last 12h):
  other_judge_or_untagged: n=70, tokens=110054, cost≈$0.046189
  openai: n=10, tokens=